In [ ]:
!pip install transformers sentence-transformers faiss-cpu datasets scikit-learn pandas numpy torch -q

In [ ]:
from google.colab import drive
drive.mount('/drive')
import os
os.makedirs('/drive/MyDrive/Recipe', exist_ok=True)
print("Drive mounted. All files will save to /drive/MyDrive/Recipe/")

In [ ]:
import pandas as pd
import numpy as np
import re
import torch
import faiss
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from torch.optim import AdamW
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from datasets import load_dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
def label_vegan(ingredients):
    non_vegan = ['chicken', 'beef', 'pork', 'lamb', 'fish', 'shrimp',
                 'egg', 'eggs', 'milk', 'cheese', 'butter', 'cream',
                 'honey', 'gelatin', 'bacon', 'turkey', 'tuna', 'salmon']
    return int(not any(w in ingredients.lower() for w in non_vegan))

def label_gluten_free(ingredients):
    gluten = ['flour', 'wheat', 'bread', 'pasta', 'barley',
              'rye', 'soy sauce', 'breadcrumbs', 'couscous', 'semolina']
    return int(not any(w in ingredients.lower() for w in gluten))

def label_diabetic(ingredients):
    high_sugar = ['sugar', 'corn syrup', 'honey', 'maple syrup',
                  'molasses', 'candy', 'chocolate syrup', 'jam']
    return int(not any(w in ingredients.lower() for w in high_sugar))

df['vegan']       = df['ingredients_text'].apply(label_vegan)
df['gluten_free'] = df['ingredients_text'].apply(label_gluten_free)
df['diabetic']    = df['ingredients_text'].apply(label_diabetic)

print("Label distribution:")
print(df[['vegan', 'gluten_free', 'diabetic']].mean().round(2))

In [ ]:
class RecipeDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], max_length=self.max_len,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.float)
        }

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

texts  = df['ingredients_text'].tolist()
labels = df[['vegan', 'gluten_free', 'diabetic']].values.tolist()

X_train, X_val, y_train, y_val = train_test_split(texts, labels, test_size=0.2, random_state=42)

train_loader = DataLoader(RecipeDataset(X_train, y_train, tokenizer), batch_size=32, shuffle=True)
val_loader   = DataLoader(RecipeDataset(X_val,   y_val,   tokenizer), batch_size=32)

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased', num_labels=3,
    problem_type='multi_label_classification'
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)

print("Training BERT...")
for epoch in range(3):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['labels'].to(device)
        optimizer.zero_grad()
        loss = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_batch).loss
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/3 | Loss: {total_loss/len(train_loader):.4f}")

model.save_pretrained('/drive/MyDrive/Recipe/bert_dietary')
tokenizer.save_pretrained('/drive/MyDrive/Recipe/bert_dietary')
print("Model saved.")

In [ ]:
model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for batch in val_loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        probs = torch.sigmoid(model(input_ids=input_ids, attention_mask=attention_mask).logits).cpu().numpy()
        all_preds.append((probs > 0.5).astype(int))
        all_labels.append(batch['labels'].numpy())

print(classification_report(
    np.vstack(all_labels), np.vstack(all_preds),
    target_names=['vegan', 'gluten_free', 'diabetic'],
    zero_division=0
))

In [ ]:
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Encoding recipes...")
recipe_embeddings = embedder.encode(
    df['ingredients_text'].tolist(),
    batch_size=256, show_progress_bar=True, convert_to_numpy=True
)
faiss.normalize_L2(recipe_embeddings)
index = faiss.IndexFlatIP(recipe_embeddings.shape[1])
index.add(recipe_embeddings)
print(f"Index built: {index.ntotal} recipes")

In [ ]:
faiss.write_index(index, '/drive/MyDrive/Recipe/recipe_index.faiss')
df.to_csv('/drive/MyDrive/Recipe/recipes_clean.csv', index=False)
print("Index and dataset saved to Drive.")

In [5]:
def predict_dietary_labels(ingredient_text):
    model.eval()
    enc = tokenizer(
        ingredient_text, max_length=64, padding='max_length',
        truncation=True, return_tensors='pt'
    ).to(device)
    with torch.no_grad():
        probs = torch.sigmoid(model(**enc).logits).cpu().numpy()[0]
    label_names = ['Vegan', 'Gluten-free', 'Diabetic-friendly']
    return [label_names[i] for i, p in enumerate(probs) if p > 0.5], probs

def recommend_recipes(ingredient_input, top_k=30, top_n=5):
    dietary_labels, probs = predict_dietary_labels(ingredient_input)
    print(f"\nInput       : {ingredient_input}")
    print(f"Dietary tags: {dietary_labels if dietary_labels else 'None detected'}")
    print(f"Confidence  : vegan={probs[0]:.2f} | gluten_free={probs[1]:.2f} | diabetic={probs[2]:.2f}")

    query_vec = embedder.encode([ingredient_input], convert_to_numpy=True)
    faiss.normalize_L2(query_vec)
    scores, indices = index.search(query_vec, top_k)

    candidates = df.iloc[indices[0]].copy()
    candidates['cosine_score'] = scores[0]

    user_tokens = set(ingredient_input.lower().replace(',', ' ').split())
    candidates['coverage_score'] = candidates['ingredients_text'].apply(
        lambda t: len(user_tokens & set(t.lower().replace(',', ' ').split())) / max(len(set(t.lower().replace(',', ' ').split())), 1)
    )

    filtered = candidates.copy()
    if 'Vegan' in dietary_labels and probs[0] > 0.7:
        filtered = filtered[filtered['vegan'] == 1]
    if 'Gluten-free' in dietary_labels and probs[1] > 0.7:
        filtered = filtered[filtered['gluten_free'] == 1]
    if 'Diabetic-friendly' in dietary_labels and probs[2] > 0.7:
        filtered = filtered[filtered['diabetic'] == 1]
    if len(filtered) < top_n:
        print("(Relaxing dietary filter)")
        filtered = candidates.copy()

    filtered['final_score'] = 0.6 * filtered['cosine_score'] + 0.4 * filtered['coverage_score']
    filtered = filtered.sort_values('final_score', ascending=False).head(top_n)

    print(f"\n{'─'*60}")
    for rank, (_, row) in enumerate(filtered.iterrows(), 1):
        tags = []
        if row['vegan']:       tags.append('Vegan')
        if row['gluten_free']: tags.append('GF')
        if row['diabetic']:    tags.append('Diabetic')
        print(f"\n{rank}. {row['title']}")
        print(f"   Score      : {row['final_score']:.3f} (cosine={row['cosine_score']:.3f}, coverage={row['coverage_score']:.3f})")
        print(f"   Tags       : {', '.join(tags) if tags else 'None'}")
        print(f"   Ingredients: {row['ingredients_text'][:80]}...")

In [ ]:
user_input = input("Enter ingredients (comma separated): ")
recommend_recipes(user_input)


In [ ]:
# Cell 1: install
!pip install transformers sentence-transformers faiss-cpu scikit-learn torch -q

# Mount drive
from google.colab import drive
drive.mount('/drive')

import pandas as pd, numpy as np, torch, faiss
from transformers import BertTokenizer, BertForSequenceClassification
from sentence_transformers import SentenceTransformer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

df       = pd.read_csv('/drive/MyDrive/Recipe/recipes_clean.csv')
tokenizer = BertTokenizer.from_pretrained('/drive/MyDrive/Recipe/bert_dietary')
model     = BertForSequenceClassification.from_pretrained('/drive/MyDrive/Recipe/bert_dietary').to(device)
model.eval()
index    = faiss.read_index('/drive/MyDrive/Recipe/recipe_index.faiss')
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print(f"Ready — {len(df)} recipes loaded, index has {index.ntotal} vectors")